# ch05 Bonus 06：替代权重加载方式

> 对照官方 `ch05/02_alternative_weight_loading`

## 一句话

主线用 `tf` 格式加载 OpenAI 权重，本 notebook 演示用 **safetensors / HF Transformers** 这类更现代、更安全的方式加载同一份 GPT-2 权重。

## 为什么要换

- **pickle 反序列化有安全风险**（可执行任意代码），`safetensors` 规避了这点
- HF 生态统一了权重格式，便于加载社区模型
- 加载速度更快、支持内存映射（mmap）

> 本 notebook 演示「权重键名映射」这个核心环节：不同保存格式的键名约定不同，需要一一对应。

In [ ]:
import torch
import torch.nn as nn
from src.gpt import GPTModel, GPT_CONFIG_124M

# 演示「键名映射」：不同格式对同一权重的命名不同
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

# 我们的模型键名（我们的命名约定）
our_keys = sorted([k for k in model.state_dict().keys() if "trf_blocks.0" in k and "weight" in k])
print("我们的键名约定（第 0 层 TransformerBlock）：")
for k in our_keys[:6]:
    print(f"  {k}")

# HuggingFace 的命名约定（键名前缀不同）
print("\nHuggingFace 命名约定（对应）：")
hf_mapping = {
    "trf_blocks.{i}.att.W_query.weight": "transformer.h.{i}.attn.c_attn.weight",  # HF 把 qkv 合并
    "trf_blocks.{i}.ff.layers.0.weight": "transformer.h.{i}.mlp.c_fc.weight",
    "trf_blocks.{i}.norm1.scale":       "transformer.h.{i}.ln_1.weight",
    "trf_blocks.{i}.norm1.shift":       "transformer.h.{i}.ln_1.bias",
}
for ours, hf in list(hf_mapping.items())[:4]:
    print(f"  我们: {ours}")
    print(f"  HF:   {hf}\n")

print("💡 核心工作是写一张这样的映射表，把外部权重对齐到我们的 state_dict。")

In [ ]:
# 演示 load_weights 的核心逻辑（键名替换 + 形状对齐）
def load_hf_weights_demo(our_state, external_state, key_map):
    """演示权重加载逻辑：把 external_state 按 key_map 填入 our_state。"""
    loaded = 0
    for our_key in our_state:
        # 查映射表找到对应的外部键名
        ext_key = key_map.get(our_key)
        if ext_key and ext_key in external_state:
            our_tensor = our_state[our_key]
            ext_tensor = external_state[ext_key]
            if our_tensor.shape == ext_tensor.shape:
                our_state[our_key] = ext_tensor.clone()
                loaded += 1
    return loaded

# 模拟：构造一份「外部权重」和映射表
our_sd = {"trf_blocks.0.ff.layers.0.weight": torch.zeros(3072, 768),
          "trf_blocks.0.norm1.scale": torch.zeros(768)}
external = {"transformer.h.0.mlp.c_fc.weight": torch.ones(3072, 768),
            "transformer.h.0.ln_1.weight": torch.ones(768)}
key_map = {
    "trf_blocks.0.ff.layers.0.weight": "transformer.h.0.mlp.c_fc.weight",
    "trf_blocks.0.norm1.scale": "transformer.h.0.ln_1.weight",
}
n = load_hf_weights_demo(our_sd, external, key_map)
print(f"成功加载 {n} 个权重")
print(f"验证 ff 层已变为全 1: {our_sd['trf_blocks.0.ff.layers.0.weight'][0,0].item()}")